
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 데모 - 문서를 구조화된 데이터로 파싱하기

## 개요

이번 데모에서는 Databricks' **AI 기반 문서 파싱** 기능을 사용해 비정형 문서를 구조화된 데이터로 파싱하는 방법을 살펴보겠습니다. 이 과정을 통해 볼륨에 저장된 파일에서 표, 이미지, 텍스트를 추출할 수 있어 정보 기술 분석, 검색, 하위 Workflows 구축이 더 쉬워집니다.

대부분의 실제 검색 에이전트 사용 사례에서는, 구조화된 지식 기반을 만들기 위해 문서를 파싱해야 하는 경우가 있습니다. 이 지식 기반은 언어 모델에 추가적인 맥락을 제공하는 데 활용될 수 있습니다.

## 학습 목표
이 데모가 끝날 때쯤이면 다음을 할 수 있게 됩니다:
- SQL과 Python 모두에서 `ai_parse_document()` AI 함수를 사용하여 다중 형식 문서(PDF, DOCX)를 **파싱**합니다.
- 파싱된 출력 스키마를 **검토**하고 이해합니다.
- 파싱된 출력에서 주요 메타데이터 필드를 **식별**하고 해석합니다.
- 파싱된 문서 내용을 **시각화**하고 디버깅합니다.

## 요구 사항:
- 샘플 문서가 포함된 볼륨. **이것은 설정 코드로 만들어졌습니다.**
- **서버리스 Compute (환경 버전 5)**. [여기](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version)에 따라 적절한 환경 버전을 선택하세요.
- 서버리스 compute 구성의 **의존성**에 필요한 라이브러리가 추가됩니다.



## 준비

다음 셀을 실행하여 교실 환경을 설정하세요. 샘플 문서를 포함하고 이 데모에 사용될 볼륨 이름이 아래에 인쇄됩니다.

In [0]:
%run ../Includes/Classroom-Setup-02

다음 단계를 완료하여 귀하의 볼륨에 저장된 PDF 파일 중 하나를 확인하세요. 위에 인쇄된 볼륨 이름을 사용하여 이 지침을 따르세요.

1. 워크스페이스 사이드바에서 *카탈로그*를 클릭하면 Data Explorer을 열어보세요.
1. 자신의 환경에 맞는 **카탈로그**를 선택하세요.
1. 카탈로그 내에서 관련 **schema (`datasets`)** 를 확장하세요.
1. **orion-docs** 볼륨을 찾아서 확장하세요.
1. **01_Orion_A1_Product_Overview.pdf** 파일을 로컬 컴퓨터에 다운로드하세요.
1. 파일을 열고 내용을 검토하여 문서를 이해한 후 구문 분석하세요.


## A. 문서 구문 분석 AI

이 섹션에서는 `ai_parse_document` AI 함수를 사용하여 비구조화 문서에서 구조화된 데이터를 추출하는 방법을 배울 것입니다. 이 기능은 Databricks Mosaic AI을 활용해 PDF나 이미지와 같은 파일에서 텍스트, 표, 이미지를 자동으로 식별하고 추출합니다. 우리는 SQL와 Python 접근법을 모두 시연하고, 파서가 생성하는 키 메타데이터 필드를 설명할 것입니다. 이 기능은 문서 처리 자동화와 고급 분석 지원에 매우 유용합니다.

**🚨 참고:** 이 데모에서 파싱된 형식을 다루려면 **버전 2** `ai_parse_document` 를 사용해야 합니다.

### A1. Python로 문서 구문 분석

**Python**는 유연하고 상호작용적인 Workflows 및 머신 러닝 또는 사용자 정의 로직과의 통합에 이상적입니다. 여기서는 Spark DataFrame API와 `expr` 함수 및 `ai_parse_document` AI 함수를 사용하여 각 파일에 대해 호출합니다. 이를 통해 결과를 검사하거나 추가 변환을 적용하거나 ML 파이프라인을 구축할 수 있습니다.

코드를 실행한 후에는 DataFrame 출력을 검토하여 각 문서가 어떻게 파싱되는지 확인하세요. 볼륨이 비어 있다면 경로와 권한 설정을 확인하세요.

`ai_parse_document` AI 함수는 다음과 같은 옵션을 수용합니다:
- **`version`**: 사용할 파서 버전(예: `'2.0'` ).
- **`imageOutputPath`**: 추출한 이미지를 어디에 저장할지.
- **`descriptionElementTypes`**: 어떤 원소를 추출할지(예: `*`, `table`, `image`, `text`).


**참고:** `display` 함수가 결과를 보여주지 않아서 이진 콘텐츠 필드를 삭제합니다. 이진 필드는 너무 길어서 표시할 수 없습니다.

In [0]:
from pyspark.sql.functions import expr

# 문서 볼륨의 모든 파일을 읽습니다.
docs_df = spark.read.format("binaryFile").load(user_docs_path)

# 각 문서를 ai_parse_document을 사용하여 SQL AI 함수를 호출하려면 expr을 사용하여 구문 분석합니다.
parsed_df = docs_df.withColumn("parsed_content", 
                               expr(f"""ai_parse_document(content, map(
                                    "version", "2.0",
                                    "imageOutputPath", "{user_docs_path}/parsed_images/"
                                   ))""")
                              )
# 이진 콘텐츠 삭제
parsed_df = parsed_df.drop("content")

# 파싱된 결과 샘플을 표시하세요
display(parsed_df)

볼륨 경로에 파일을 나열하세요. 

  이 폴더에는 데모에 사용된 원본 PDF 파일과 함께, 파싱 과정에서 생성된 모든 출력 이미지를 보관하는 새로운 `/parsed_images/` 디렉터리가 포함되어 있음을 확인하십시오.

In [0]:
spark.sql(f"LIST '{user_docs_path}'").display()

아래 셀을 실행해 보면, 볼륨 내 `/parsed_images/` 디렉터리에 이제 일련의 출력 이미지가 포함되어 있음을 관찰해 보세요.

In [0]:
spark.sql(f"LIST '{user_docs_path}/parsed_images'").display()

### A2. SQL 파싱

**SQL**는 배치 처리와 Lakehouse 테이블과의 쉽게 통합하기에 좋습니다. 아래 예시는 지정된 권의 모든 문서를 구문 분석하고 구조화된 결과를 반환합니다.

이 방법은 예약된 작업이나 결과를 테이블에 유지하고 싶을 때 이상적입니다.

**참고:** 이미지 출력 경로가 정의되지 않은 경우, 이전 파싱 중 추출된 이미지도 파싱됩니다. 

In [0]:
parsed_df_sql = spark.sql(f"""
SELECT
  path,
  ai_parse_document(
    content,
    map(
      'version', '2.0'
    )
  ) as parsed_doc
FROM read_files('{user_docs_path}', format => 'binaryFile')""")

display(parsed_df_sql)

### A3. 파싱된 문서 메타데이터 이해하기

출력 `ai_parse_document`은 `parsed_content` 필드 내에서 풍부한 메타데이터 구조를 포함합니다. **Key 필드는 다음과 같습니다:**

- **`parsed:document:pages`**: 각 페이지 객체의 목록:
  - **`page_number`**: 문서 내 페이지 색인.
  - **`text`**: 추출된 텍스트 내용.
  - **`tables`**: 구조화된 테이블 데이터, 감지된 경우.
  - **`images`**: 추출된 이미지, 주로 base64나 파일 참조 형태로 제공됩니다.
- **`parsed:document:metadata`**: 일반 문서 정보(파일 이름, 크기, 형식).

*이 필드를 이용해 검색 인덱스를 만들거나, 보고 자동화하거나, 하위 ML 모델을 입력할 수 있습니다. 큰 문서의 경우, 페이지 지정이나 결과를 필터링하는 것을 고려하세요. 일부 필드가 빠졌다면 문서 유형과 파서 옵션을 확인하세요.*

In [0]:
# 문서 경로 표시와 키 메타데이터 필드
from pyspark.sql.functions import expr

# 중첩된 필드에 대해 expr로 키 메타데이터 필드를 선택하세요
meta_df = parsed_df.select(
    "path",
    expr("parsed_content:document:pages"),
    expr("parsed_content:document:elements"),
    expr("parsed_content:error_status"),
    expr("parsed_content:corrupted_data"),
    expr("parsed_content:metadata")
)

display(meta_df)

## B. 보기 그리고 파싱된 문서 내용 디버그

파싱 후에는 출력물을 점검하고 디버깅하여 품질과 완전성을 보장하는 것이 중요합니다. **시각화**는 추출 정확도를 검증하고 파서가 혼합 콘텐츠를 어떻게 처리하는지 이해하는 데 도움을 줍니다. 우리는 도우미 클래스를 사용해 파싱된 결과를 렌더링할 것이며, 그것이 각 페이지의 텍스트, 표, 이미지를 더 쉽게 검토할 수 있도록 할 것입니다.

### B1. DocumentRenderer 헬퍼 클래스 가져오기

이 `DocumentRenderer` 클래스는 Databricks Notebooks로 파싱된 문서 내용을 시각화하는 유틸리티입니다. 그것은 각 페이지에서 텍스트, 테이블, 이미지를 렌더링하는 기능을 지원하며, 이는 문서 QA와 workflow 검증에 매우 중요합니다.

헬퍼 클래스는 **Includes** 폴더에서 찾거나 강사가 제공한 곳에서 찾을 수 있습니다. 여기서는 구현 방법에 대해서는 다루지 않겠습니다 — 그냥 작업 흐름을 간소화하고 결과 해석에 집중하는 데 활용하세요.

In [0]:
# DocumentRenderer 헬퍼 클래스를 가져오기
import sys, os
sys.path.append(os.path.abspath('..'))
from Includes.document_renderer import render_ai_parse_output, render_ai_parse_output_interactive

### B2. 파싱된 결과 표시

아래 코드는 이미지와 테이블(가능하다면)이 포함된 문서를 선택하고, 헬퍼 클래스를 `DocumentRenderer` 사용해 파싱된 내용을 시각화합니다. 문서 페이지의 시각적 표현, 추출된 텍스트, 표, 이미지 등을 볼 수 있을 것입니다. 이를 통해 AI 파서가 하류 분석을 위해 콘텐츠를 어떻게 구조화했는지 디버깅하고 이해하는 것이 훨씬 쉬워집니다.

*적절한 문서를 찾지 못하면, 혼합된 내용이 있는 파일이 있는지 볼륨을 확인하거나 필터 로직을 조정하세요.*

In [0]:
# 샘플 문서를 선택하여 render_ai_parse_output을 사용하여 파싱된 내용을 렌더링합니다.
sample = parsed_df.select("parsed_content").limit(1).collect()

if sample:
    doc = sample[0]["parsed_content"]
    render_ai_parse_output(doc)
else:
    print("No parsed documents found. Please check your input volume and parsing step.")

## C. 파싱된 결과를 Delta 테이블에 저장

이제 문서를 분석하고 탐색했으니, 결과를 더 자세히 처리하기 위해 저장해 둡니다. 파싱된 콘텐츠는 현재 JSON 형식이므로, 이를 효과적으로 검색하기 전에 **콘텐츠를 정리하고 변환해야 합니다.**

In [0]:
# 파싱된 결과를 Delta 테이블로 저장해 쉽게 쿼리하고 공유하세요
output_table = f"{catalog}.{schema}.docs_parsed"

# 그것이 이미 존재한다면 테이블을 덮어쓰기
parsed_df.write.format("delta").mode("overwrite").saveAsTable(output_table)

print(f"✅ Parsed results saved to Delta table: {output_table}")

## 요약과 다음 단계

당신은 Python과 SQL 모두에서 Databricks의 AI 기반 `ai_parse_document` AI 함수를 사용해 비정형 문서를 파싱하는 방법을 배웠습니다. 우리는 배치 파일 처리, 구조화된 콘텐츠와 메타데이터 추출, 품질 보증을 위한 결과 시각화 방법을 시연했습니다. 이러한 기법을 통합함으로써 문서 추출 워크플로를 자동화하고 하위 분석 또는 머신 러닝 작업을 위한 데이터를 준비할 수 있습니다.

**핵심 포인트:**
- Python과 SQL 모두에서 `ai_parse_document` AI 함수를 사용하여 문서 파싱을 **자동화**하십시오.
- 구문 분석된 출력 스키마를 **검사**하고 이해하며, 여기에는 `pages`, `elements`, `metadata`와 같은 주요 메타데이터 필드가 포함됩니다.
- 품질 보증 및 워크플로 검증 목적으로 DocumentRenderer 헬퍼 클래스를 사용하여 구문 분석된 결과를 **시각화**하고 디버깅합니다.

`ai_parse_document`에 대한 자세한 내용은 [공식 Databricks 문서](https://docs.databricks.com/sql/language-manual/functions/ai_parse_document.html)를 참조하십시오.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>